# Playbook 06 — Candidate Ranking & Validation

Stitch together the full hybrid QML-KG repurposing pipeline:

1. Pull KG + QML scores for candidate compound-disease pairs.
2. Attach reversal scores from the perturbation layer.
3. Fuse evidence (kg-only vs kg+omics).
4. Run clinical / literature validation on the top-N.
5. Write a ranked report.

This mirrors `scripts/run_full_repurposing_pipeline.py` interactively.

In [ ]:
import logging, json
from pathlib import Path
logging.basicConfig(level=logging.INFO)
import numpy as np
import pandas as pd

## 1. Build candidate set

In production these come from `scripts/run_optimized_pipeline.py`. Here we use the demo seed set so the playbook runs without external state.

In [ ]:
from evidence_layer.evidence_schema import EvidenceFeatures
candidates = [
    EvidenceFeatures(
        compound='metformin', compound_hetionet_id='Compound::DB00331',
        disease='type 2 diabetes mellitus', disease_hetionet_id='Disease::DOID:9352',
        kg_rotate_score=0.91, qsvc_score=0.80, classical_ensemble_score=0.88,
        signature_reversal_score=0.75, cell_type_reversal_score=0.70, pathway_reversal_score=0.68,
        clinical_evidence_score=1.0,
    ),
    EvidenceFeatures(
        compound='aspirin', compound_hetionet_id='Compound::DB00945',
        disease='diabetes mellitus', disease_hetionet_id='Disease::DOID:9351',
        kg_rotate_score=0.85, qsvc_score=0.72, classical_ensemble_score=0.79,
        signature_reversal_score=0.68, cell_type_reversal_score=0.61, pathway_reversal_score=0.55,
        clinical_evidence_score=0.5,
    ),
    EvidenceFeatures(
        compound='rosuvastatin', compound_hetionet_id='Compound::DB01098',
        disease='hyperlipidemia', disease_hetionet_id='Disease::DOID:1387',
        kg_rotate_score=0.79, qsvc_score=0.66, classical_ensemble_score=0.74,
        signature_reversal_score=0.55, cell_type_reversal_score=0.48, pathway_reversal_score=0.40,
        clinical_evidence_score=1.0,
    ),
]
print(f'{len(candidates)} candidates')

## 2. Fuse evidence — KG-only vs KG+omics

In [ ]:
from evidence_layer.feature_fusion import fuse_evidence
from evidence_layer.explanation_builder import attach_explanations
import copy

kg_only = copy.deepcopy(candidates)
fused_kg = fuse_evidence(kg_only, mode='kg-only')

kg_omics = copy.deepcopy(candidates)
fused_omics = fuse_evidence(kg_omics, mode='kg+omics')
attach_explanations(fused_omics)

print('kg-only top:', fused_kg[0].compound,    f'{fused_kg[0].final_score:.4f}',    f'tier={fused_kg[0].confidence_tier}')
print('kg+omics top:', fused_omics[0].compound, f'{fused_omics[0].final_score:.4f}', f'tier={fused_omics[0].confidence_tier}')

## 3. Per-candidate evidence breakdown

In [ ]:
print(fused_omics[0].explanation)

## 4. Validation — known indications & clinical trials

In [ ]:
from validation_layer.known_indications_validator import check_known_indication
from validation_layer.drugbank_mapper import check_drugbank_indication, get_drugbank_indications
for ef in fused_omics:
    known = check_known_indication(ef.compound_hetionet_id, ef.disease_hetionet_id)
    db_ind = check_drugbank_indication(ef.compound_hetionet_id, ef.disease)
    print(f'{ef.compound:18s}  known={known}  drugbank={db_ind}  evidence_score={ef.clinical_evidence_score}')

## 5. Write final report

In [ ]:
from evidence_layer.evidence_report import write_evidence_report
out = write_evidence_report(fused_omics, out_dir='artifacts/predictions', top_n=10, disease_id='Disease::DOID:9352')
print(f'Report: {out}')

## 6. Compare modes — KG-only vs KG+omics

Compute the ΔPR-AUC contribution of omics on a synthetic held-out set. In production this comes from `scripts/run_full_repurposing_pipeline.py --mode {kg-only,kg+omics}`.

In [ ]:
import pandas as pd
comp = pd.DataFrame({
    'compound': [ef.compound for ef in fused_kg],
    'kg_only':  [ef.final_score for ef in fused_kg],
    'kg_omics': [ef.final_score for ef in sorted(fused_omics, key=lambda x: [c.compound for c in fused_kg].index(x.compound))],
})
comp['delta'] = comp['kg_omics'] - comp['kg_only']
comp